# Roman Strong Lens Data Challenge — Rung 0
## ViT Two-Stage Pipeline — Classification then Regression

This notebook builds a two-stage pipeline for simulated Roman Space Telescope
images of strong gravitational lenses, using **Vision Transformers** for both
stages:

1. **Stage 1 — Classifier**: a ViT that decides whether an image is a lens or a nonlens.
2. **Stage 2 — Regressor**: a ViT that predicts the **Einstein radius** (θ_e), trained on the ground-truth lens images. The cascade at the end still feeds it whatever Stage 1 flags as a lens — false positives included — so the end-to-end evaluation reflects real-world pipeline behavior.

### Pipeline overview
1. Load the dataset from the `.h5` file
2. Build color images from the three infrared filters (F106, F129, F158)
3. Train the Stage 1 ViT classifier on lens vs. nonlens images
4. Train the Stage 2 ViT regressor on the ground-truth lens images
5. Chain them together: classify first, then regress only what got classified as a lens — regardless of whether that classification is correct
6. Evaluate performance at each stage, and end-to-end


## Imports and setup

In [ ]:
import math
import h5py
import numpy as np
import matplotlib.pyplot as plt
from astropy.visualization import make_lupton_rgb

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

from sklearn.metrics import r2_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Check if a GPU is available — this is what your professor's machine provides
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU name: {torch.cuda.get_device_name(0)}')

## Load the dataset

In [ ]:
# ── UPDATE THIS PATH ──────────────────────────────────────────────────────────
h5_filepath = 'roman_data_challenge_rung_0_6040split_v_1_0.h5'
# ─────────────────────────────────────────────────────────────────────────────

f = h5py.File(h5_filepath, 'r')

# The images are stored under a group — find its name
print('Top-level keys in file:', list(f.keys()))

In [ ]:
images_group = f['images']

In [ ]:
print(f'Number of lens systems: {len(images_group.keys())}')

first_key = list(images_group.keys())[0]
first_group = images_group[first_key]
print(f'\nGroup-level attributes for first system: {list(first_group.attrs.keys())}')
print(f'Datasets inside first system: {list(first_group.keys())}')

In [ ]:
from challenge_data import attr_val, get_bool_attr_val

sample_keys = list(images_group.keys())[:10]
for k in sample_keys:
    grp = images_group[k]
    deflector_only = get_bool_attr_val(attr_val(grp.attrs, 'deflector_only'))
    theta_e = float(attr_val(grp.attrs, 'theta_e'))
    print(f"{k}: deflector_only={deflector_only}, theta_e={theta_e:.4f}")

## Build image tensors and labels

In [ ]:
# NOTE: these are the ground-truth lens images, used only for the preview
# plot below. The regressor's actual training data (images/labels) gets
# built LATER, from whatever the trained classifier flags as a lens —
# see the "Build the regressor's dataset from the classifier" cell in Stage 2.
all_lens_images = []
all_lens_labels = []
all_lens_rgb_previews = []
all_lens_snr_values = []

from challenge_data import get_filter_dataset, normalize

for strong_lens_group in images_group.keys():
    lens = images_group[strong_lens_group]

    is_strong_lens = not get_bool_attr_val(attr_val(lens.attrs, 'deflector_only'))
    if not is_strong_lens:
        continue  # this preview/regression set only wants actual lensed systems

    einstein_radius = float(attr_val(lens.attrs, 'theta_e'))

    ds_r = get_filter_dataset(lens, 'F158')
    ds_g = get_filter_dataset(lens, 'F129')
    ds_b = get_filter_dataset(lens, 'F106')

    raw_r = ds_r[:]
    raw_g = ds_g[:]
    raw_b = ds_b[:]

    snr = float(attr_val(ds_b.attrs, 'snr'))  # pulled from the F106 exposure

    norm_r = normalize(raw_r)
    norm_g = normalize(raw_g)
    norm_b = normalize(raw_b)

    stacked = np.stack([norm_r, norm_g, norm_b], axis=0)
    all_lens_images.append(stacked)
    all_lens_labels.append(einstein_radius)
    all_lens_snr_values.append(snr)

    rgb = make_lupton_rgb(image_r=norm_r, image_g=norm_g, image_b=norm_b,
                          stretch=0.2, Q=4)
    all_lens_rgb_previews.append(rgb)

all_lens_images = np.array(all_lens_images, dtype=np.float32)
all_lens_labels = np.array(all_lens_labels, dtype=np.float32)
all_lens_snr_values = np.array(all_lens_snr_values, dtype=np.float32)

print(f'Images array shape : {all_lens_images.shape}')
print(f'Labels array shape : {all_lens_labels.shape}')
print(f'SNR — min: {all_lens_snr_values.min():.2f}, max: {all_lens_snr_values.max():.2f}')

In [ ]:
all_nonlens_images = []
all_nonlens_rgb_previews = []
all_nonlens_snr_values = []

for strong_lens_group in images_group.keys():
    lens = images_group[strong_lens_group]

    is_strong_lens = not get_bool_attr_val(attr_val(lens.attrs, 'deflector_only'))
    if is_strong_lens:
        continue  # this set only wants non-lensed (deflector-only) systems

    ds_r = get_filter_dataset(lens, 'F158')
    ds_g = get_filter_dataset(lens, 'F129')
    ds_b = get_filter_dataset(lens, 'F106')

    raw_r = ds_r[:]
    raw_g = ds_g[:]
    raw_b = ds_b[:]

    snr = float(attr_val(ds_b.attrs, 'snr'))  # pulled from the F106 exposure

    norm_r = normalize(raw_r)
    norm_g = normalize(raw_g)
    norm_b = normalize(raw_b)

    stacked = np.stack([norm_r, norm_g, norm_b], axis=0)
    all_nonlens_images.append(stacked)
    all_nonlens_snr_values.append(snr)

    rgb = make_lupton_rgb(image_r=norm_r, image_g=norm_g, image_b=norm_b,
                          stretch=0.2, Q=4)
    all_nonlens_rgb_previews.append(rgb)

all_nonlens_images = np.array(all_nonlens_images, dtype=np.float32)
all_nonlens_snr_values = np.array(all_nonlens_snr_values, dtype=np.float32)

print(f'Images array shape : {all_nonlens_images.shape}')
print(f'SNR — min: {all_nonlens_snr_values.min():.2f}, max: {all_nonlens_snr_values.max():.2f}')

In [ ]:
images_clf = []
labels_clf = []  # 1.0 = lens, 0.0 = nonlens
theta_e_clf = []  # true radius, carried along for the cascade eval later
snr_clf = []      # true SNR, carried along the same way

for strong_lens_group in images_group.keys():
    group = images_group[strong_lens_group]

    is_strong_lens = not get_bool_attr_val(attr_val(group.attrs, 'deflector_only'))
    einstein_radius = float(attr_val(group.attrs, 'theta_e'))

    ds_r = get_filter_dataset(group, 'F158')
    ds_g = get_filter_dataset(group, 'F129')
    ds_b = get_filter_dataset(group, 'F106')

    raw_r = ds_r[:]
    raw_g = ds_g[:]
    raw_b = ds_b[:]

    snr = float(attr_val(ds_b.attrs, 'snr'))

    norm_r = normalize(raw_r)
    norm_g = normalize(raw_g)
    norm_b = normalize(raw_b)

    stacked = np.stack([norm_r, norm_g, norm_b], axis=0)
    images_clf.append(stacked)

    labels_clf.append(1.0 if is_strong_lens else 0.0)
    theta_e_clf.append(einstein_radius)
    snr_clf.append(snr)

images_clf  = np.array(images_clf, dtype=np.float32)
labels_clf  = np.array(labels_clf, dtype=np.float32)
theta_e_clf = np.array(theta_e_clf, dtype=np.float32)
snr_clf     = np.array(snr_clf, dtype=np.float32)

print(f'Images_clf shape : {images_clf.shape}')
print(f'Labels_clf shape : {labels_clf.shape}')
print(f'Positive (lens) count : {int(labels_clf.sum())} / {len(labels_clf)}')

# Stage 1. Classify lenses and non-lenses

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from challenge_evaluation import snr_stratified_sample

grid_size = 4
n_show = grid_size ** 2  # 16

lens_sample_idx = snr_stratified_sample(all_lens_snr_values, n_show)
nonlens_sample_idx = snr_stratified_sample(all_nonlens_snr_values, n_show)

lens_imgs = [all_lens_rgb_previews[i] for i in lens_sample_idx]
lens_ers  = [all_lens_labels[i] for i in lens_sample_idx]
lens_snrs = [all_lens_snr_values[i] for i in lens_sample_idx]

nonlens_imgs = [all_nonlens_rgb_previews[i] for i in nonlens_sample_idx]
nonlens_snrs = [all_nonlens_snr_values[i] for i in nonlens_sample_idx]

# --- Plot ---
fig, axes = plt.subplots(grid_size, grid_size * 2, figsize=(16, 8), constrained_layout=True)

for i in range(n_show):
    row, col = i // grid_size, i % grid_size

    # Non-lenses on the left
    ax_nl = axes[row, col]
    ax_nl.imshow(nonlens_imgs[i])
    ax_nl.set_title(f'SNR = {nonlens_snrs[i]:.1f}', fontsize=12)
    ax_nl.axis('off')

    # Lenses on the right
    ax_l = axes[row, col + grid_size]
    ax_l.imshow(lens_imgs[i])
    ax_l.set_title(f'θ_e = {lens_ers[i]:.2f}"  SNR = {lens_snrs[i]:.1f}', fontsize=12)
    ax_l.axis('off')

# Column-group headers
fig.text(0.27, 1.02, 'Non-Lenses (spanning full SNR range)', ha='center', fontsize=13, fontweight='bold')
fig.text(0.75, 1.02, 'Lenses (spanning full SNR range)', ha='center', fontsize=13, fontweight='bold')

plt.suptitle('Sample Non-Lenses vs Lenses — Lupton RGB (F158/F129/F106)', fontsize=14, y=1.06)
plt.show()

# Also plot the distribution of Einstein radii
plt.figure(figsize=(8, 4))
plt.hist(all_lens_labels, bins=50, color='steelblue', edgecolor='white')
plt.xlabel('Einstein Radius (arcseconds)')
plt.ylabel('Count')
plt.title('Distribution of Einstein Radii in Dataset')
plt.tight_layout()
plt.show()

## Augment classifier images

## Build classifier data loaders

In [ ]:
from challenge_data import augment_dataset

images_clf_aug, labels_clf_aug = augment_dataset(images_clf, labels_clf)

print(f'Original classifier dataset  : {images_clf.shape[0]:,} images')
print(f'Augmented classifier dataset : {images_clf_aug.shape[0]:,} images  (8× via dihedral symmetry)')

## Define the vision transformer

In [ ]:
class LensDataset(Dataset):
    '''Simple PyTorch Dataset wrapping our numpy arrays.'''
    def __init__(self, images, labels):
        self.images = torch.tensor(images, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]


full_dataset_clf = LensDataset(images_clf_aug, labels_clf_aug)
total_clf = len(full_dataset_clf)

n_train_clf = int(0.70 * total_clf)
n_val_clf   = int(0.15 * total_clf)
n_test_clf  = total_clf - n_train_clf - n_val_clf

train_set_clf, val_set_clf, test_set_clf = random_split(
    full_dataset_clf, [n_train_clf, n_val_clf, n_test_clf],
    generator=torch.Generator().manual_seed(42)
)

BATCH_SIZE = 64

train_loader_clf = DataLoader(train_set_clf, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader_clf   = DataLoader(val_set_clf,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader_clf  = DataLoader(test_set_clf,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Training samples   : {n_train_clf:,}')
print(f'Validation samples : {n_val_clf:,}')
print(f'Test samples       : {n_test_clf:,}')

## Configure classifier training

In [ ]:
class PatchEmbedding(nn.Module):
    '''
    Small convolutional stem + patchify, instead of a single linear
    projection straight from raw pixels.

    The stem (stride-1, 3x3 convs) lets the model build up basic local
    texture/edge sensitivity — the kind of low-level features
    a convolutional model would get from its own first conv layer — *before* the image gets
    chopped into patches and handed to self-attention. This gives the
    ViT a bit of convolutional inductive bias up front, which helps
    convergence and generalization on smaller datasets like this one.

    Spatial size is preserved through the stem (stride=1, padding=1),
    so padding-to-patch-multiple still happens exactly as before, just
    after the stem instead of directly on the raw image.
    '''
    def __init__(self, in_channels=3, patch_size=9, embed_dim=128,
                 img_size=73, stem_channels=32):
        super().__init__()
        self.patch_size = patch_size

        # ── Convolutional stem: 3 stride-1 conv layers, no downsampling ──
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, stem_channels, kernel_size=3,
                      stride=1, padding=1, bias=False),
            nn.BatchNorm2d(stem_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(stem_channels, stem_channels, kernel_size=3,
                      stride=1, padding=1, bias=False),
            nn.BatchNorm2d(stem_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(stem_channels, stem_channels, kernel_size=3,
                      stride=1, padding=1, bias=False),
            nn.BatchNorm2d(stem_channels),
            nn.ReLU(inplace=True),
        )

        # Compute padded size (next multiple of patch_size >= img_size)
        self.padded_size = math.ceil(img_size / patch_size) * patch_size
        pad_total = self.padded_size - img_size
        self.pad_left = pad_total // 2
        self.pad_right = pad_total - self.pad_left

        self.grid_size = self.padded_size // patch_size
        self.num_patches = self.grid_size * self.grid_size

        # Patchify: projects from stem_channels (not raw 3) to embed_dim
        self.proj = nn.Conv2d(stem_channels, embed_dim,
                               kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.stem(x)              # (batch, stem_channels, 73, 73)
        x = F.pad(x, (self.pad_left, self.pad_right,
                       self.pad_left, self.pad_right))
        x = self.proj(x)              # (batch, embed_dim, grid, grid)
        x = x.flatten(2)              # (batch, embed_dim, num_patches)
        x = x.transpose(1, 2)         # (batch, num_patches, embed_dim)
        return x


class TransformerNet(nn.Module):
    '''
    Vision Transformer — used for BOTH stages. The final layer is a single
    raw number (Linear(64, 1), no activation). For the regressor that
    number IS the predicted theta_e. For the classifier, that same number
    is treated as a logit and passed through BCEWithLogitsLoss/sigmoid.

    Architecture:
        Input        : (batch, 3, 73, 73)
        Patch embed  : conv stem -> pad to 81x81 -> 9x9 patches -> embed_dim
        CLS token    : learnable token prepended to the sequence
        Pos embed    : learnable positional embedding added to every token
        Transformer  : `depth` stacked encoder blocks (self-attn + MLP)
        Head         : CLS token's output -> MLP -> scalar
    '''
    def __init__(self, in_channels=3, img_size=73, patch_size=9,
                 embed_dim=96, depth=4, num_heads=6, mlp_ratio=4.0,
                 dropout=0.1):
        super().__init__()

        self.patch_embed = PatchEmbedding(in_channels, patch_size, embed_dim, img_size)
        num_patches = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.pos_drop = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=int(embed_dim * mlp_ratio),
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True   # pre-LN blocks train more stably than post-LN
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        self.norm = nn.LayerNorm(embed_dim)

        self.fc_head = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(64, 1)
        )

        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, x):
        batch_size = x.shape[0]

        x = self.patch_embed(x)                       # (batch, num_patches, embed_dim)

        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)          # (batch, num_patches+1, embed_dim)
        x = x + self.pos_embed
        x = self.pos_drop(x)

        x = self.transformer(x)
        x = self.norm(x)

        cls_output = x[:, 0]                           # take CLS token only
        out = self.fc_head(cls_output)
        return out.squeeze(1)


# ── Instantiate the Stage 1 classifier ViT ───────────────────────────────
clf_model = TransformerNet(
    in_channels=3, img_size=91, patch_size=9,
    embed_dim=96, depth=4, num_heads=6, mlp_ratio=4.0, dropout=0.1
).to(device)
print(clf_model)

total_params = sum(p.numel() for p in clf_model.parameters() if p.requires_grad)
print(f'\nTotal trainable parameters: {total_params:,}')

## Train the classifier

In [ ]:
LEARNING_RATE_CLF = 3e-4
NUM_EPOCHS_CLF    = 150

criterion_clf = nn.BCEWithLogitsLoss()
optimizer_clf = optim.AdamW(clf_model.parameters(), lr=LEARNING_RATE_CLF, weight_decay=0.05)

scheduler_clf = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_clf, mode='min', factor=0.5, patience=9
)

WARMUP_EPOCHS_CLF = 5

def warmup_lambda_clf(epoch):
    if epoch < WARMUP_EPOCHS_CLF:
        return (epoch + 1) / WARMUP_EPOCHS_CLF
    return 1.0

warmup_scheduler_clf = optim.lr_scheduler.LambdaLR(optimizer_clf, lr_lambda=warmup_lambda_clf)

class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float('inf')
        self.counter = 0
        self.should_stop = False

    def step(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
        return self.should_stop


early_stopper_clf = EarlyStopping(patience=18, min_delta=1e-5)

print(f'Loss      : BCEWithLogitsLoss')
print(f'Optimizer : AdamW (lr={LEARNING_RATE_CLF}, weight_decay=0.05)')
print(f'Max Epochs: {NUM_EPOCHS_CLF}  (warmup={WARMUP_EPOCHS_CLF} epochs, '
      f'scheduler patience={scheduler_clf.patience}, early-stop patience={early_stopper_clf.patience})')

## Evaluate classifier results on test data

In [ ]:
train_losses_clf = []
val_losses_clf   = []
best_val_loss_clf = float('inf')

early_stopper_clf.best_loss = float('inf')
early_stopper_clf.counter = 0
early_stopper_clf.should_stop = False

for epoch in range(NUM_EPOCHS_CLF):
    clf_model.train()
    running_train_loss = 0.0
    running_train_correct = 0

    for images_batch, labels_batch in train_loader_clf:
        images_batch = images_batch.to(device)
        labels_batch = labels_batch.to(device)

        optimizer_clf.zero_grad()
        logits = clf_model(images_batch)
        loss = criterion_clf(logits, labels_batch)
        loss.backward()
        optimizer_clf.step()

        running_train_loss += loss.item() * len(images_batch)
        preds = (torch.sigmoid(logits) > 0.5).float()
        running_train_correct += (preds == labels_batch).sum().item()

    avg_train_loss = running_train_loss / n_train_clf
    avg_train_acc  = running_train_correct / n_train_clf

    clf_model.eval()
    running_val_loss = 0.0
    running_val_correct = 0

    with torch.no_grad():
        for images_batch, labels_batch in val_loader_clf:
            images_batch = images_batch.to(device)
            labels_batch = labels_batch.to(device)
            logits = clf_model(images_batch)
            loss = criterion_clf(logits, labels_batch)
            running_val_loss += loss.item() * len(images_batch)
            preds = (torch.sigmoid(logits) > 0.5).float()
            running_val_correct += (preds == labels_batch).sum().item()

    avg_val_loss = running_val_loss / n_val_clf
    avg_val_acc  = running_val_correct / n_val_clf

    train_losses_clf.append(avg_train_loss)
    val_losses_clf.append(avg_val_loss)

    if avg_val_loss < best_val_loss_clf:
        best_val_loss_clf = avg_val_loss
        torch.save(clf_model.state_dict(), 'best_clf_model_vit.pt')

    # During warmup, step warmup_scheduler_clf. After warmup, hand off to
    # ReduceLROnPlateau — same pattern as the ViT regressor's training loop.
    old_lr = optimizer_clf.param_groups[0]['lr']
    if epoch < WARMUP_EPOCHS_CLF:
        warmup_scheduler_clf.step()
    else:
        scheduler_clf.step(avg_val_loss)
    new_lr = optimizer_clf.param_groups[0]['lr']
    if new_lr != old_lr:
        tag = 'Warmup' if epoch < WARMUP_EPOCHS_CLF else 'Learning rate reduced'
        print(f'  → {tag}: {old_lr:.2e} → {new_lr:.2e}')

    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1:3d}/{NUM_EPOCHS_CLF} — '
              f'Train Loss: {avg_train_loss:.4f}  Train Acc: {avg_train_acc:.4f}  |  '
              f'Val Loss: {avg_val_loss:.4f}  Val Acc: {avg_val_acc:.4f}  |  '
              f'EarlyStop counter: {early_stopper_clf.counter}/{early_stopper_clf.patience}')

    if early_stopper_clf.step(avg_val_loss):
        print(f'\nEarly stopping triggered at epoch {epoch+1}.')
        print(f'Best validation loss: {early_stopper_clf.best_loss:.6f}')
        break

print(f'\nBest validation loss: {best_val_loss_clf:.6f}')

# Stage 2. Estimate the Einstein radius

In [ ]:
clf_model.load_state_dict(torch.load('best_clf_model_vit.pt', map_location=device))
clf_model.eval()

all_preds_clf  = []
all_probs_clf  = []
all_labels_clf = []

with torch.no_grad():
    for images_batch, labels_batch in test_loader_clf:
        images_batch = images_batch.to(device)
        logits = clf_model(images_batch)
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float()

        all_preds_clf.extend(preds.cpu().numpy())
        all_probs_clf.extend(probs.cpu().numpy())
        all_labels_clf.extend(labels_batch.numpy())

all_preds_clf  = np.array(all_preds_clf)
all_probs_clf  = np.array(all_probs_clf)
all_labels_clf = np.array(all_labels_clf)

test_acc  = accuracy_score(all_labels_clf, all_preds_clf)
test_prec = precision_score(all_labels_clf, all_preds_clf)
test_rec  = recall_score(all_labels_clf, all_preds_clf)
test_f1   = f1_score(all_labels_clf, all_preds_clf)
cm        = confusion_matrix(all_labels_clf, all_preds_clf)

print('── Stage 1 Test Set Results ──────────────────────')
print(f'Accuracy  : {test_acc:.4f}')
print(f'Precision : {test_prec:.4f}')
print(f'Recall    : {test_rec:.4f}')
print(f'F1        : {test_f1:.4f}')
print('──────────────────────────────────────────────────')
print('Confusion matrix (rows=true, cols=pred), order = [nonlens, lens]:')
print(cm)

## Select training images from the true lens sample

## Augment regressor images

In [ ]:
images = all_lens_images
labels = all_lens_labels
rgb_previews = all_lens_rgb_previews
snr_values = all_lens_snr_values

print(f'Regressor training set (ground-truth lenses): {images.shape}')


## Build regressor data loaders

In [ ]:
images_aug, labels_aug = augment_dataset(images, labels)

print(f'Original dataset  : {images.shape[0]:,} images')
print(f'Augmented dataset : {images_aug.shape[0]:,} images  (8× via dihedral symmetry)')

## Split the regressor dataset

In [ ]:
full_dataset = LensDataset(images_aug, labels_aug)
total = len(full_dataset)

n_train = int(0.70 * total)
n_val   = int(0.15 * total)
n_test  = total - n_train - n_val

train_set, val_set, test_set = random_split(
    full_dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42)   # reproducible split
)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Training samples   : {n_train:,}')
print(f'Validation samples : {n_val:,}')
print(f'Test samples       : {n_test:,}')

## Instantiate the vision transformer regressor

In [ ]:
model = TransformerNet(
    in_channels=3, img_size=91, patch_size=9,
    embed_dim=96, depth=4, num_heads=6, mlp_ratio=4.0, dropout=0.1
).to(device)
print(model)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal trainable parameters: {total_params:,}')

---
## Regressor training setup

Identical to your original ViT regressor's setup:
**Loss:** Huber loss (delta=0.1) — behaves like MSE for small errors, like MAE for large ones, so a few huge outliers can't dominate the gradient.
**Optimizer:** AdamW, lr `3e-4`, weight_decay `0.05`. **Warmup:** linear ramp over 5 epochs. **Scheduler:** halves the LR after 9 stagnant epochs (post-warmup). **Early stopping:** stops after 18 stagnant epochs.

---

In [ ]:
LEARNING_RATE = 3e-4
NUM_EPOCHS    = 150

HUBER_DELTA = 0.1

criterion = nn.HuberLoss(delta=HUBER_DELTA)

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.05)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=9
)

WARMUP_EPOCHS = 5

def warmup_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    return 1.0

warmup_scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=warmup_lambda)

early_stopper = EarlyStopping(patience=18, min_delta=1e-5)

print(f'Loss      : Huber (delta={HUBER_DELTA})')
print(f'Optimizer : AdamW (lr={LEARNING_RATE}, weight_decay=0.05)')
print(f'Max Epochs: {NUM_EPOCHS}  (warmup={WARMUP_EPOCHS} epochs, '
      f'scheduler patience={scheduler.patience}, early-stop patience={early_stopper.patience})')
print(f'Batch     : {BATCH_SIZE}')

---
## Train the regressor

Trains `model` and saves the best weights (by validation loss) to `best_model_vit.pt`. This must run before the cascade cells at the end of the notebook — they load these saved weights rather than retraining.

---

In [ ]:
train_losses = []
val_losses   = []
best_val_loss = float('inf')

early_stopper.best_loss = float('inf')
early_stopper.counter = 0
early_stopper.should_stop = False

for epoch in range(NUM_EPOCHS):

    model.train()
    running_train_loss = 0.0

    for images_batch, labels_batch in train_loader:
        images_batch = images_batch.to(device)
        labels_batch = labels_batch.to(device)

        optimizer.zero_grad()
        predictions = model(images_batch)
        loss = criterion(predictions, labels_batch)
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * len(images_batch)

    avg_train_loss = running_train_loss / n_train

    model.eval()
    running_val_loss = 0.0

    with torch.no_grad():
        for images_batch, labels_batch in val_loader:
            images_batch = images_batch.to(device)
            labels_batch = labels_batch.to(device)
            predictions  = model(images_batch)
            loss = criterion(predictions, labels_batch)
            running_val_loss += loss.item() * len(images_batch)

    avg_val_loss = running_val_loss / n_val

    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'best_model_vit.pt')

    old_lr = optimizer.param_groups[0]['lr']
    if epoch < WARMUP_EPOCHS:
        warmup_scheduler.step()
    else:
        scheduler.step(avg_val_loss)
    new_lr = optimizer.param_groups[0]['lr']
    if new_lr != old_lr:
        tag = 'Warmup' if epoch < WARMUP_EPOCHS else 'Learning rate reduced'
        print(f'  → {tag}: {old_lr:.2e} → {new_lr:.2e}')

    if (epoch + 1) % 5 == 0:
        train_rmse = np.sqrt(avg_train_loss)
        val_rmse   = np.sqrt(avg_val_loss)
        print(f'Epoch {epoch+1:3d}/{NUM_EPOCHS} — '
              f'Train RMSE: {train_rmse:.4f}"  |  '
              f'Val RMSE: {val_rmse:.4f}"  |  '
              f'EarlyStop counter: {early_stopper.counter}/{early_stopper.patience}')

    if early_stopper.step(avg_val_loss):
        print(f'\nEarly stopping triggered at epoch {epoch+1} '
              f'(no improvement for {early_stopper.patience} epochs).')
        print(f'Best validation loss (MSE): {early_stopper.best_loss:.6f}')
        break

print(f'\nBest validation loss (MSE): {best_val_loss:.6f}')
print(f'Best validation RMSE      : {np.sqrt(best_val_loss):.4f} arcseconds')

---
## Regressor training history

---

In [ ]:
epochs = range(1, epoch+2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, train_losses, label='Train Loss (Huber)', color='steelblue')
axes[0].plot(epochs, val_losses,   label='Val Loss (Huber)',   color='tomato')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training vs Validation Loss')
axes[0].legend()

axes[1].plot(epochs, np.sqrt(train_losses), label='Train RMSE', color='steelblue')
axes[1].plot(epochs, np.sqrt(val_losses),   label='Val RMSE',   color='tomato')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('RMSE (arcseconds)')
axes[1].set_title('Training vs Validation RMSE')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## Evaluate the regressor on its test set

---

In [ ]:
model.load_state_dict(torch.load('best_model_vit.pt', map_location=device))
model.eval()

all_preds  = []
all_labels = []

with torch.no_grad():
    for images_batch, labels_batch in test_loader:
        images_batch = images_batch.to(device)
        preds = model(images_batch)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels_batch.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

test_mse  = np.mean((all_preds - all_labels) ** 2)
test_rmse = np.sqrt(test_mse)
test_mae  = np.mean(np.abs(all_preds - all_labels))
test_r2   = r2_score(all_labels, all_preds)

print('── Stage 2 Test Set Results ──────────────────────')
print(f'RMSE : {test_rmse:.4f} arcseconds')
print(f'MAE  : {test_mae:.4f} arcseconds')
print(f'R²   : {test_r2:.4f}')
print('──────────────────────────────────────────────────')
print()
print('Interpretation:')
print(f'  On average, predictions are off by ~{test_rmse:.3f} arcseconds')
print(f'  R² of {test_r2:.3f} means the model explains {test_r2*100:.1f}% of the variance in Einstein radius')

---
## Regressor predictions

Predicted vs true scatter, and residuals — how good is the regressor's own test-set performance? This looks only at Stage 2 in isolation, using the test split from its own training data (the ground-truth lenses). It does NOT yet account for the classifier's mistakes — that end-to-end view comes later, in the Two-Stage Cascade section.

---

In [ ]:


fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(all_labels, all_preds, alpha=0.3, s=8, color='steelblue')
lims = [all_labels.min(), all_labels.max()]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
axes[0].set_xlabel('True Einstein Radius (arcseconds)')
axes[0].set_ylabel('Predicted Einstein Radius (arcseconds)')
axes[0].set_title(f'Predicted vs True  (R² = {test_r2:.3f})')
axes[0].legend()

residuals = all_preds - all_labels
axes[1].scatter(all_labels, residuals, alpha=0.3, s=8, color='steelblue')
axes[1].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('True Einstein Radius (arcseconds)')
axes[1].set_ylabel('Residual: Predicted − True (arcseconds)')
axes[1].set_title('Residuals vs True Einstein Radius')

plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.hist(residuals, bins=60, color='steelblue', edgecolor='white')
plt.axvline(0, color='red', linestyle='--')
plt.xlabel('Residual (arcseconds)')
plt.ylabel('Count')
plt.title('Distribution of Prediction Residuals')
plt.tight_layout()
plt.show()

In [ ]:
# ── Error vs SNR ─────────────────────────────────────────────────────────────

test_indices = test_set.indices
orig_indices_full = np.array([idx // 8 for idx in test_indices])

test_snr = snr_values[orig_indices_full]
abs_error = np.abs(all_preds - all_labels)

theta_e_median = np.median(all_labels)
is_large_theta_e = all_labels >= theta_e_median

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sc = axes[0].scatter(test_snr, abs_error, c=all_labels, cmap='viridis',
                      alpha=0.5, s=12)
axes[0].set_xlabel('Signal-to-Noise Ratio')
axes[0].set_ylabel('Absolute Error (arcsec)')
axes[0].set_xscale('log')
axes[0].set_title('Error vs SNR, colored by true θ_e')
cbar = plt.colorbar(sc, ax=axes[0])
cbar.set_label('True θ_e (arcsec)')

n_bins = 8
snr_bin_edges = np.percentile(test_snr, np.linspace(0, 100, n_bins + 1))
snr_bin_centers = 0.5 * (snr_bin_edges[:-1] + snr_bin_edges[1:])

for mask, label, color in [(~is_large_theta_e, f'Small θ_e (< {theta_e_median:.2f}")', 'steelblue'),
                             (is_large_theta_e, f'Large θ_e (≥ {theta_e_median:.2f}")', 'crimson')]:
    binned_median_error = []
    for i in range(n_bins):
        in_bin = mask & (test_snr >= snr_bin_edges[i]) & (test_snr < snr_bin_edges[i + 1])
        if in_bin.sum() > 0:
            binned_median_error.append(np.median(abs_error[in_bin]))
        else:
            binned_median_error.append(np.nan)
    axes[1].plot(snr_bin_centers, binned_median_error, 'o-', color=color, label=label)

from matplotlib.ticker import LogLocator, NullFormatter

axes[1].set_xlabel('Signal-to-Noise Ratio (binned)')
axes[1].set_ylabel('Median Absolute Error (arcsec)')
axes[1].set_xscale('log')
axes[1].set_title('Median error vs SNR — flat line = irreducible error')

# limit to major ticks only, and space them out
axes[1].xaxis.set_minor_formatter(NullFormatter())  # hide minor tick labels
axes[1].xaxis.set_major_locator(LogLocator(base=10, numticks=6))
axes[1].tick_params(axis='x', labelsize=15, rotation=0)

axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
from challenge_evaluation import (
    attach_physical_properties,
    get_worst_predictions,
    plot_worst_images,
    print_worst_summary,
)

In [ ]:
worst_df = get_worst_predictions(all_preds, all_labels, orig_indices_full, n_worst=25)
worst_df = attach_physical_properties(worst_df, labels, snr_values=snr_values)
print_worst_summary(worst_df, n_show=25)
plot_worst_images(worst_df, rgb_previews, n_show=12, cols=3, panel_size=5)


In [ ]:
theta_e_bin_edges = np.percentile(all_labels, [0, 25, 50, 75, 100])
theta_e_bin_edges[-1] += 0.01
theta_e_bin_labels = ['Q1 (smallest θ_e)', 'Q2', 'Q3', 'Q4 (largest θ_e)']

print(f'Test set θ_e range: {all_labels.min():.3f}" to {all_labels.max():.3f}"')
print(f'Quartile edges: {theta_e_bin_edges}')

print('\n── R² and Error Metrics by Einstein Radius Quartile ────────────────────')
print(f'{"Bin":<20} {"Range (arcsec)":<20} {"N":>6} {"RMSE":>8} {"MAE":>8} {"R²":>8}')
print('─' * 75)

bin_results = []
for i in range(len(theta_e_bin_edges) - 1):
    lo, hi = theta_e_bin_edges[i], theta_e_bin_edges[i + 1]
    mask = (all_labels >= lo) & (all_labels < hi)
    n_in_bin = mask.sum()

    if n_in_bin < 5:
        print(f'{theta_e_bin_labels[i]:<20} {"":<20} {n_in_bin:>6}    (too few samples)')
        continue

    bin_preds  = all_preds[mask]
    bin_truth  = all_labels[mask]

    bin_rmse = np.sqrt(np.mean((bin_preds - bin_truth) ** 2))
    bin_mae  = np.mean(np.abs(bin_preds - bin_truth))
    bin_r2   = r2_score(bin_truth, bin_preds)

    range_str = f'{lo:.3f}–{hi:.3f}'
    bin_results.append({
        'bin': theta_e_bin_labels[i], 'range': range_str, 'n': n_in_bin,
        'rmse': bin_rmse, 'mae': bin_mae, 'r2': bin_r2
    })

    print(f'{theta_e_bin_labels[i]:<20} {range_str:<20} {n_in_bin:>6} {bin_rmse:>8.4f} {bin_mae:>8.4f} {bin_r2:>8.4f}')

print('─' * 75)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bins_plotted = [r['bin']  for r in bin_results]
r2_plotted   = [r['r2']   for r in bin_results]
rmse_plotted = [r['rmse'] for r in bin_results]
mae_plotted  = [r['mae']  for r in bin_results]

axes[0].bar(bins_plotted, r2_plotted, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Einstein Radius Quartile')
axes[0].set_ylabel('R²')
axes[0].set_title('R² by Einstein Radius (θ_e) Quartile')
axes[0].tick_params(axis='x', rotation=15)

x = np.arange(len(bins_plotted))
width = 0.35
axes[1].bar(x - width/2, rmse_plotted, width, label='RMSE', color='tomato')
axes[1].bar(x + width/2, mae_plotted,  width, label='MAE',  color='steelblue')
axes[1].set_xticks(x)
axes[1].set_xticklabels(bins_plotted, rotation=15)
axes[1].set_xlabel('Einstein Radius Quartile')
axes[1].set_ylabel('Error (arcseconds)')
axes[1].set_title('RMSE vs MAE by Einstein Radius (θ_e) Quartile')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## Inspect individual regressor predictions

---

In [ ]:
test_indices = test_set.indices
orig_indices = [idx // 8 for idx in test_indices[:25]]

fig, axes = plt.subplots(5, 5, figsize=(14, 14), constrained_layout=True)

for i, (orig_idx, pred, true) in enumerate(zip(orig_indices, all_preds[:25], all_labels[:25])):
    row, col = i // 5, i % 5
    axes[row, col].imshow(rgb_previews[orig_idx])
    axes[row, col].set_title(
        f'True: {true:.2f}"\nPred: {pred:.2f}"',
        fontsize=8,
        color='green' if abs(pred - true) < 0.05 else 'red'
    )
    axes[row, col].axis('off')

plt.suptitle('Test Set Predictions  (green = within 0.05", red = larger error)', fontsize=13)
plt.show()

In [ ]:
THRESHOLD = 0.05

correct = np.abs(all_preds - all_labels) < THRESHOLD
pct_correct = 100 * np.mean(correct)

print(f'Correct (within {THRESHOLD}"): {correct.sum()} / {len(all_labels)}')
print(f'Percentage correct: {pct_correct:.2f}%')

---
# Two-stage cascade

This is the actual end-to-end pipeline: every test image goes through `clf_model` first. Only the images `clf_model` classifies as lenses get passed to `model` for a θ_e prediction. Images classified as nonlens never reach the regressor at all — they get `NaN` for `theta_e`.

This requires both training loops above (Cell 9 for the classifier, Cell 15 for the regressor) to have already been run, since it just loads whichever weights are saved to `best_clf_model_vit.pt` and `best_model_vit.pt`.

---

In [ ]:
from challenge_evaluation import run_two_stage_pipeline

In [ ]:
test_indices_clf = test_set_clf.indices
cascade_images = images_clf_aug[test_indices_clf]
cascade_true_is_lens = labels_clf_aug[test_indices_clf]

# theta_e wasn't augmented, so map back to the pre-augmentation index
# (aug index // 8 = original images_clf index) to look up the true radius
orig_positions = [idx // 8 for idx in test_indices_clf]
cascade_true_theta_e = theta_e_clf[orig_positions]

cascade_tensor = torch.tensor(cascade_images, dtype=torch.float32)

pred_is_lens, pred_theta_e = run_two_stage_pipeline(cascade_tensor, clf_model, model, device)

print(f'Ran pipeline on {len(cascade_images)} test images')
print(f'Stage 1 classified {int(pred_is_lens.sum())} as lenses, '
      f'{len(pred_is_lens) - int(pred_is_lens.sum())} as nonlens')


In [ ]:
acc  = accuracy_score(cascade_true_is_lens, pred_is_lens)
prec = precision_score(cascade_true_is_lens, pred_is_lens)
rec  = recall_score(cascade_true_is_lens, pred_is_lens)
f1   = f1_score(cascade_true_is_lens, pred_is_lens)
cm   = confusion_matrix(cascade_true_is_lens, pred_is_lens)

print('── Stage 1: Classifier Performance ───────────────────────')
print(f'Accuracy  : {acc:.4f}')
print(f'Precision : {prec:.4f}')
print(f'Recall    : {rec:.4f}')
print(f'F1        : {f1:.4f}')
print('Confusion matrix (rows=true, cols=pred), order = [nonlens, lens]:')
print(cm)

true_lens_mask = cascade_true_is_lens == 1
pred_lens_mask = pred_is_lens == 1
true_positive_mask = true_lens_mask & pred_lens_mask

n_true_lens = int(true_lens_mask.sum())
print(f'\n── Pipeline Breakdown ──────────────────────────────')
print(f'True lenses in test set        : {n_true_lens}')
print(f'  → passed through to Stage 2  : {int(true_positive_mask.sum())} '
      f'({100*true_positive_mask.sum()/n_true_lens:.1f}%)')
print(f'  → missed at Stage 1          : {int((true_lens_mask & ~pred_lens_mask).sum())}')
print(f'Nonlens wrongly passed to Stage 2: {int(((~true_lens_mask) & pred_lens_mask).sum())}')

rmse = np.sqrt(np.mean((pred_theta_e[pred_lens_mask] - cascade_true_theta_e[pred_lens_mask]) ** 2))
mae  = np.mean(np.abs(pred_theta_e[pred_lens_mask] - cascade_true_theta_e[pred_lens_mask]))
r2   = r2_score(cascade_true_theta_e[pred_lens_mask], pred_theta_e[pred_lens_mask])

n_pred_lens = int(pred_lens_mask.sum())
n_false_pos = int((~true_lens_mask & pred_lens_mask).sum())
print(f'\n── Stage 2: Regression Performance (on all {n_pred_lens} images Stage 1 classified as lenses) ──')
print(f'  Of these, {n_false_pos} are classifier false positives '
      f'(true nonlens, scored against their deflector θ_e anyway)')
print(f'RMSE : {rmse:.4f} arcseconds')
print(f'MAE  : {mae:.4f} arcseconds')
print(f'R²   : {r2:.4f}')


---
## Summary

### What this notebook does
| Step | Description |
|---|---|
| Data loading | Opens the `.h5` file, builds a lens+nonlens preview set and a lens+nonlens classifier dataset |
| Stage 1 | Trains `clf_model` (`TransformerNet` ViT + `BCEWithLogitsLoss`) to flag lens vs. nonlens |
| Stage 2 dataset | The ground-truth lens images from Cell 3 — kept separate from whatever the classifier flags |
| Stage 2 | Trains `model` (`TransformerNet` ViT + Huber loss) to predict θ_e on the ground-truth lens images |
| Cascade | Chains them: classify every test image first, regress only the ones flagged as lenses — false positives included, since the regressor never sees Stage 1's classification decision, only the image |
| Evaluation | Accuracy/precision/recall/F1 for Stage 1; RMSE/MAE/R² for Stage 2; end-to-end breakdown for the full pipeline |

### Architecture
`TransformerNet` — a small convolutional stem (3 stride-1 conv layers) for local texture/edge features, followed by a patchify projection, a learnable `[CLS]` token, learnable positional embeddings, and `depth` stacked pre-LN transformer encoder blocks. Reused for both stages; only the loss function and output interpretation change. Optimizer: `AdamW` + linear LR warmup over the first few epochs, then `ReduceLROnPlateau` — transformers are more sensitive to a too-high initial LR than CNNs, since they lack convolutional inductive bias to stabilize early gradients.

### Run order
Run every cell top to bottom once. The two training loops (Stage 1's and Stage 2's) each take a while and only need to run once per session — after that, the saved `best_clf_model_vit_6040.pt`/`best_model_vit_6040.pt` weights are what the cascade cells at the end actually use.

### If Stage 1 recall is low
Real lenses are being lost before they ever reach the regressor — check the confusion matrix in Cell 10 and consider lowering the classification `threshold` in `run_two_stage_pipeline` below 0.5 to trade some precision for recall.
